In [ ]:
import os
import copy
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm


TRAIN_HR_DIR = "/kaggle/input/datasets/haliten/33halitsen-espcn-dataset/train/hr"
TRAIN_LR_DIR = "/kaggle/input/datasets/haliten/33halitsen-espcn-dataset/train/lr"

VALID_HR_DIR = "/kaggle/input/datasets/haliten/33halitsen-espcn-dataset/test/hr_1080p"
VALID_LR_DIR = "/kaggle/input/datasets/haliten/33halitsen-espcn-dataset/test/lr_360p"

OUTPUT_DIR = "/kaggle/working/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "fast_res_espcn_best.pth")
CSV_SAVE_PATH = os.path.join(OUTPUT_DIR, "training_history_fast_res.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
UPSCALE_FACTOR = 3

BATCH_SIZE = 32 
LEARNING_RATE = 1e-4
MAX_EPOCHS = 1000
EARLY_STOP_PATIENCE = 15


class DynamicYChannelDataset(Dataset):
    def __init__(self, hr_dir, lr_dir, is_train=True, lr_patch_size=48):
        self.hr_dir = hr_dir
        self.lr_dir = lr_dir
        self.is_train = is_train
        self.lr_patch_size = lr_patch_size
        self.hr_patch_size = lr_patch_size * UPSCALE_FACTOR
        
        self.image_filenames = [f for f in os.listdir(hr_dir) if f.endswith((".png", ".jpg"))]
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        hr_image = Image.open(os.path.join(self.hr_dir, img_name)).convert("YCbCr")
        lr_image = Image.open(os.path.join(self.lr_dir, img_name)).convert("YCbCr")

        if self.is_train:
            w, h = lr_image.size
            x = random.randint(0, max(0, w - self.lr_patch_size))
            y = random.randint(0, max(0, h - self.lr_patch_size))
            
            lr_image = lr_image.crop((x, y, x + self.lr_patch_size, y + self.lr_patch_size))
            hr_image = hr_image.crop((x * UPSCALE_FACTOR, y * UPSCALE_FACTOR, 
                                      x * UPSCALE_FACTOR + self.hr_patch_size, 
                                      y * UPSCALE_FACTOR + self.hr_patch_size))
            
            if random.random() > 0.5:
                lr_image = transforms.functional.hflip(lr_image)
                hr_image = transforms.functional.hflip(hr_image)

        y_lr, _, _ = lr_image.split()
        y_hr, _, _ = hr_image.split()

        return self.transform(y_lr), self.transform(y_hr)


class FastResidualBlock(nn.Module):
    def __init__(self, channels):
        super(FastResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = x
        out = self.relu(self.conv1(x))
        out = self.conv2(out)
        return out + residual

class FastResESPCN(nn.Module):
    def __init__(self, upscale_factor=3):
        super(FastResESPCN, self).__init__()
        
        self.conv_in = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        
        self.res_block1 = FastResidualBlock(64)
        self.res_block2 = FastResidualBlock(64)
        
        self.conv_out = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        self.conv_ps = nn.Conv2d(32, 1 * (upscale_factor**2), kernel_size=3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(upscale_factor)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.conv_in(x))
        x = self.res_block1(x)
        x = self.res_block2(x)
        x = self.relu(self.conv_out(x))
        x = self.conv_ps(x)
        x = self.pixel_shuffle(x)
        return x


class Trainer:
    def __init__(self, model, train_loader, valid_loader, min_delta=1e-4):
        self.model = model.to(DEVICE)
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.min_delta = min_delta 

        self.criterion = nn.L1Loss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode="min", factor=0.5, patience=5)
        
        self.best_val_loss = float("inf")
        self.epochs_no_improve = 0
        self.best_model_wts = copy.deepcopy(self.model.state_dict())
        self.history = {"Epoch": [], "Train_Loss": [], "Valid_Loss": [], "Learning_Rate": []}

    def train(self):
        print(f"\n[{DEVICE.type.upper()}] Fast-Res-ESPCN Eğitimi Başlıyor!")
        for epoch in range(MAX_EPOCHS):
            self.model.train()
            train_loss = 0.0
            train_bar = tqdm(self.train_loader, desc=f"Epoch {epoch+1} [Train]")
            for lr_imgs, hr_imgs in train_bar:
                lr_imgs, hr_imgs = lr_imgs.to(DEVICE), hr_imgs.to(DEVICE)
                self.optimizer.zero_grad()
                outputs = self.model(lr_imgs)
                loss = self.criterion(outputs, hr_imgs)
                loss.backward()
                self.optimizer.step()
                train_loss += loss.item() * lr_imgs.size(0)
                train_bar.set_postfix({"L1_Loss": f"{loss.item():.4f}"})
            avg_train_loss = train_loss / len(self.train_loader.dataset)

            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for lr_imgs, hr_imgs in self.valid_loader:
                    lr_imgs, hr_imgs = lr_imgs.to(DEVICE), hr_imgs.to(DEVICE)
                    outputs = self.model(lr_imgs)
                    loss = self.criterion(outputs, hr_imgs)
                    val_loss += loss.item() * lr_imgs.size(0)
            avg_val_loss = val_loss / len(self.valid_loader.dataset)

            current_lr = self.optimizer.param_groups[0]["lr"]
            print(f"-> Train: {avg_train_loss:.6f} | Valid: {avg_val_loss:.6f} | LR: {current_lr}")

            self.history["Epoch"].append(epoch + 1)
            self.history["Train_Loss"].append(avg_train_loss)
            self.history["Valid_Loss"].append(avg_val_loss)
            self.history["Learning_Rate"].append(current_lr)

            self.scheduler.step(avg_val_loss)

            if avg_val_loss < (self.best_val_loss - self.min_delta):
                self.best_val_loss = avg_val_loss
                self.epochs_no_improve = 0
                self.best_model_wts = copy.deepcopy(self.model.state_dict())
            else:
                self.epochs_no_improve += 1

            if self.epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"\n🛑 early stoping: {self.best_val_loss:.6f}")
                break

        self._save_results()

    def _save_results(self):
        self.model.load_state_dict(self.best_model_wts)
        torch.save(self.model.state_dict(), MODEL_SAVE_PATH)
        df = pd.DataFrame(self.history)
        df.to_csv(CSV_SAVE_PATH, index=False)

if __name__ == "__main__":
    train_dataset = DynamicYChannelDataset(TRAIN_HR_DIR, TRAIN_LR_DIR, is_train=True)
    
    valid_dataset = DynamicYChannelDataset(VALID_HR_DIR, VALID_LR_DIR, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    
    valid_loader = DataLoader(valid_dataset, batch_size=1, shuffle=False, num_workers=2)

    model = FastResESPCN(upscale_factor=UPSCALE_FACTOR)
    trainer = Trainer(model, train_loader, valid_loader, min_delta=5e-5)
    trainer.train()